# K-MRC 흐름 수정: 교체할 셀
독립 실행용이 아닙니다. 00(설정)/04/05/06/08/15/20/21/28의 기존 코드 셀을 교체하고 커널 재시작 후 00번부터 실행합니다. 00번 import 셀은 유지하고 개인 조회 조건은 옮기세요.


## 00. 교체할 코드
동일 step 값이면 단일 조회·배율 1. 별도 edge step의 RAWDATA X/Y에만 기본 배율 0.5. 단일 step의 K-MRC용 OCO E1에만 144000 µm 반경 제한을 적용하며 ADI는 제한하지 않습니다.


In [ ]:
@dataclass
class PipelineConfig:
    days: int = 8
    exclude_recent_days: int = 1
    expo_days: int = 10
    target_lotid: list[str] | str | None = None
    target_pstepseq: str | None = "VC075030"
    target_mstepseq: str | None = "VC075040"
    target_mstepseq_oco: str | None = "VC077251"
    include_oco_edge: bool = True
    target_mstepseq_oco_edge: str | None = "VC077252"
    oco_edge_scale: float = 0.5
    kmrc_e1_radius_um: float = 144000.0
    db_user: str = "SIMAXP2"
    db_user_apc: str = "EAPCP2"
    slot_min: int = 9
    slot_max: int = 14
    rawdata_gap_seconds: int = 60
    apc_match_seconds: int = 60
    wafer_radius_um: float = 147000.0
    max_query_keys: int = 1000
    rawdata_row_limit: int = 10000000
    reference_date: str | None = None  # Python 측 ADI 날짜 필터용. SQL의 now()는 DB 기준.
    output_dir: str = "output_decorrect_notebook"
    save_parquet: bool = True
    save_excel: bool = False
    max_excel_rows: int = 1048575

    def __post_init__(self):
        if not np.isfinite(self.kmrc_e1_radius_um) or self.kmrc_e1_radius_um <= 0:
            raise ValueError("kmrc_e1_radius_um은 유한한 양수여야 합니다.")
        if not np.isfinite(self.oco_edge_scale) or self.oco_edge_scale < 0:
            raise ValueError("oco_edge_scale은 유한한 0 이상의 수여야 합니다.")
        if self.include_oco_edge and not self.target_mstepseq_oco_edge:
            raise ValueError("Edge 측정 step을 지정하세요. 기존 step과 같으면 단일 step으로 처리합니다.")
        if self.days <= 0 or not 0 <= self.exclude_recent_days < self.days:
            raise ValueError("days > exclude_recent_days >= 0 이어야 합니다.")
        if self.slot_min > self.slot_max or self.max_query_keys <= 0 or self.rawdata_row_limit <= 0:
            raise ValueError("wafer 범위와 조회 제한을 확인하세요.")
        if self.rawdata_gap_seconds < 0 or self.apc_match_seconds < 0:
            raise ValueError("시간 간격은 0 이상이어야 합니다.")

OUTPUT_DIR = Path.cwd() / "output_decorrect_notebook"
SAVE_PARQUET = True
SAVE_EXCEL = True
EXCEL_PREVIEW_ROWS = 10000  # 이보다 큰 테이블은 Excel 샘플 + 전체 Parquet을 저장
PREVIEW_ROWS = 10

cfg = PipelineConfig()
cfg.days = 8
cfg.exclude_recent_days = 1
cfg.expo_days = 10
cfg.target_lotid = None  # 예: ["B0K486.1", "B0K271.1"]
cfg.target_pstepseq = "VC075030"
cfg.target_mstepseq = "VC075040"
cfg.target_mstepseq_oco = "VC077251"
cfg.include_oco_edge = True  # False이면 기존 OCO만 사용
cfg.target_mstepseq_oco_edge = "VC077252"
cfg.kmrc_e1_radius_um = 144000.0  # 단일 step의 OCO E1 회귀에만 적용; ADI는 제한하지 않음
cfg.oco_edge_scale = 0.5  # Edge RAWDATA의 value_x/value_y에만 조회 직후 적용
cfg.slot_min = 9
cfg.slot_max = 14
cfg.reference_date = None  # Python ADI 날짜 필터. SQL now()의 날짜를 바꾸지는 않음
cfg.output_dir = str(OUTPUT_DIR)
cfg.save_parquet = SAVE_PARQUET
cfg.save_excel = SAVE_EXCEL
cfg.__post_init__()
if EXCEL_PREVIEW_ROWS < 1 or PREVIEW_ROWS < 1:
    raise ValueError("미리보기 행 수는 1 이상이어야 합니다.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("중간 결과 저장 폴더:", OUTPUT_DIR.resolve())


## 04. 교체할 코드
기존 ADI/OCO 선정 후, 서로 다른 edge step이 있을 때만 동일 LOT/wafer/photo의 edge 이력을 추가 조회합니다. 동일 step은 중복 조회하지 않습니다.


In [ ]:
def select_lot_pair(adi, oco, cfg):
    if adi.empty or oco.empty:
        return adi.iloc[:0].copy(), oco.iloc[:0].copy(), pd.DataFrame()
    adi, oco = adi.copy(), oco.copy()
    for frame in (adi, oco):
        require_columns(frame, ["photo_transn_seq", "slotid", "lot_transn_seq", "lotid", "timestamp"], "LOTINFO")
        frame["photo_transn_seq"], frame["slotid"] = integer(frame["photo_transn_seq"]), integer(frame["slotid"])
    adi["timestamp"] = pd.to_datetime(adi["timestamp"], errors="coerce")
    today = pd.Timestamp(cfg.reference_date).normalize() if cfg.reference_date else pd.Timestamp.today().normalize()
    start, end = today - pd.Timedelta(days=cfg.days), today - pd.Timedelta(days=cfg.exclude_recent_days)
    adi = adi[adi["timestamp"].ge(start) & adi["timestamp"].lt(end)].copy()
    keys = ["photo_transn_seq", "slotid"]
    common = oco[keys].dropna().drop_duplicates().merge(adi[keys].dropna().drop_duplicates(), on=keys, how="inner")
    adi, oco = [frame.merge(common, on=keys, how="inner").sort_values(keys).reset_index(drop=True) for frame in (adi, oco)]
    adi, oco = [frame[frame["slotid"].between(cfg.slot_min, cfg.slot_max)].copy() for frame in (adi, oco)]
    for frame in (adi, oco):
        frame["slotid"] = frame["slotid"].astype(int)
    oco_extra = oco[keys + ["lot_transn_seq", "timestamp", "mstepseq"]].drop_duplicates(keys)
    oco_extra = oco_extra.rename(columns={"lot_transn_seq": "lot_transn_seq_oco", "timestamp": "timestamp_oco", "mstepseq": "mstepseq_oco"})
    saved = adi.merge(oco_extra, on=keys, how="left")
    if "per_shot_mrc_id" in saved:
        parts = saved["per_shot_mrc_id"].astype(str).str.split("/").str[-1].str.split("_")
        saved["base_eqpid"] = parts.str[0]
        saved["psm_update_time"] = parts.str[1].astype("string").str.replace(r"^20", "", regex=True).str[:10]
    return adi, oco, saved

df_lotinfo_adi, df_lotinfo_oco, df_lotinfo_save = select_lot_pair(df_lotinfo_adi_query, df_lotinfo_oco_query, cfg)
save_stage("04_LOTINFO_ADI_SELECTED", df_lotinfo_adi)
save_stage("04_LOTINFO_OCO_SELECTED", df_lotinfo_oco)
save_stage("04_LOTINFO_PAIRED", df_lotinfo_save)

# OCO edge input extension: matched to the selected main wafers
def oco_edge_lotinfo_sql(cfg, selected):
    """Keep the selected exposure in the SQL before choosing its latest edge history."""
    photos = selected["photo_transn_seq"].dropna().unique().tolist()
    lots = selected["lotid"].dropna().unique().tolist()
    recent_days = max(0, cfg.exclude_recent_days - 3)
    return f"""
WITH edge_lotinfo AS (
  SELECT {LOT_COLUMNS}, row_number() OVER (
    PARTITION BY l.lotid, l.slotid, l.photo_transn_seq ORDER BY l.impala_insert_time DESC
  ) AS rn
  FROM ees_ds_eai.apc_nautil_dat_lotinfo l
  WHERE l.impala_insert_time >= now() - interval {cfg.days + 5} days
    AND l.`timestamp` >= now() - interval {cfg.days + 5} days
    AND l.`timestamp` <= now() - interval {recent_days} days
    AND l.db_user = {quote(cfg.db_user)}
    {equals_filter('l.pstepseq', cfg.target_pstepseq)}
    {equals_filter('l.mstepseq', cfg.target_mstepseq_oco_edge)}
    {in_filter('l.lotid', lots)}
    {in_filter('l.photo_transn_seq', photos, True)}
)
SELECT * FROM edge_lotinfo WHERE rn = 1
"""


def select_oco_edge_history(edge, selected):
    keys = ["lotid", "slotid", "photo_transn_seq"]
    base, edge = selected.copy(), edge.copy()
    if base.empty:
        base = base.reindex(columns=list(dict.fromkeys(list(base.columns) + keys + ["lot_transn_seq"])))
    if edge.empty:
        edge = pd.DataFrame(columns=list(dict.fromkeys(list(base.columns) + list(edge.columns))))
    for frame in [base, edge]:
        require_columns(frame, keys + ["lot_transn_seq"], "OCO EDGE LOTINFO")
        frame["lotid"] = frame["lotid"].astype("string")
        for col in ["slotid", "photo_transn_seq", "lot_transn_seq"]:
            frame[col] = integer(frame[col])
    anchor = base[keys + ["lot_transn_seq"]].dropna(subset=keys).drop_duplicates(keys)
    anchor = anchor.rename(columns={"lot_transn_seq": "main_lot_transn_seq"})
    edge = edge.dropna(subset=keys + ["lot_transn_seq"]).merge(anchor[keys], on=keys, how="inner", validate="m:1")
    if "impala_insert_time" in edge:
        edge["_edge_order"] = pd.to_datetime(edge["impala_insert_time"], errors="coerce", format="mixed")
        edge = edge.sort_values("_edge_order", na_position="first").drop(columns="_edge_order")
    edge = edge.drop_duplicates(keys, keep="last").reset_index(drop=True)
    matched = anchor.merge(edge[keys + ["lot_transn_seq"]].rename(columns={"lot_transn_seq": "edge_lot_transn_seq"}), on=keys, how="left", validate="1:1")
    matched["edge_history_found"] = matched["edge_lot_transn_seq"].notna()
    matched["edge_history_status"] = np.where(matched["edge_history_found"], "matched", "missing_edge_history")
    return edge, matched


OCO_SPLIT_STEPS = bool(cfg.include_oco_edge and cfg.target_mstepseq_oco_edge != cfg.target_mstepseq_oco)
edge_query_parts = []
if OCO_SPLIT_STEPS and not df_lotinfo_oco.empty:
    edge_photos = df_lotinfo_oco["photo_transn_seq"].dropna().unique().tolist()
    for offset in range(0, len(edge_photos), cfg.max_query_keys):
        subset = df_lotinfo_oco[df_lotinfo_oco["photo_transn_seq"].isin(edge_photos[offset:offset + cfg.max_query_keys])]
        sql_lotinfo_oco_edge = oco_edge_lotinfo_sql(cfg, subset)
        edge_query_parts.append(fetch_frame(f"lotinfo_oco_edge_{offset // cfg.max_query_keys}", sql_lotinfo_oco_edge))
df_lotinfo_oco_edge_query = pd.concat(edge_query_parts, ignore_index=True) if edge_query_parts else df_lotinfo_oco.iloc[:0].copy()
df_lotinfo_oco_edge, df_oco_edge_history_status = select_oco_edge_history(df_lotinfo_oco_edge_query, df_lotinfo_oco)
if not OCO_SPLIT_STEPS:
    df_oco_edge_history_status["edge_history_status"] = "single_step" if cfg.include_oco_edge else "disabled"
oco_history_parts = [frame for frame in [df_lotinfo_oco, df_lotinfo_oco_edge] if not frame.empty]
df_lotinfo_oco_all = pd.concat(oco_history_parts, ignore_index=True) if oco_history_parts else df_lotinfo_oco_edge.copy()
if df_lotinfo_oco_all.duplicated(["lot_transn_seq", "slotid"]).any():
    raise ValueError("기존/edge LOTINFO의 lot_transn_seq + slotid가 중복됩니다. 별도 측정 이력인지 확인하세요.")
save_stage("04_LOTINFO_OCO_EDGE_QUERY", df_lotinfo_oco_edge_query)
save_stage("04_LOTINFO_OCO_EDGE_SELECTED", df_lotinfo_oco_edge)
save_stage("04_LOTINFO_OCO_WITH_EDGE", df_lotinfo_oco_all)
save_stage("04_OCO_EDGE_HISTORY_STATUS", df_oco_edge_history_status)


## 05. 교체할 코드
기존 OCO E1/E2를 보존합니다. 별도 edge step은 최종 분석 그룹(077: E2, 그 외: E1)만 조회·결합하며 원래 X/Y와 적용 배율을 보존합니다.


In [ ]:
def rawdata_sql(cfg, lots, test_group=None):
    ids = limited_keys(lots["lot_transn_seq"], cfg.max_query_keys, "RAWDATA")
    bounds = {"E1": (1, 80), "E2": (81, 160), "E3": (161, 240)}
    group_filter = "" if test_group is None else f"AND r.test_point_no BETWEEN {bounds[test_group][0]} AND {bounds[test_group][1]}"
    return f"""
SELECT r.lot_transn_seq,r.slot_id,r.transn_occur_date,r.data_kind,r.test_point_no,r.coordinate_x,r.coordinate_y,r.value_x,r.value_y,r.mrc_x_valn,r.mrc_y_valn
FROM ees_ds_eai.apc_nautil_dat_rawdata r
WHERE r.impala_insert_time >= now() - interval {cfg.days + 10} days AND r.db_user = {quote(cfg.db_user)}
  {in_filter('r.lot_transn_seq', ids, True)} AND r.data_kind IN ('RAWDATA','TESTDATA','PERSHOT')
  {group_filter}
LIMIT {cfg.rawdata_row_limit}
"""

def prepare_oco_raw_source(raw, source_name, mstepseq, scale=1.0):
    """Scale only RAWDATA.value_x/y once, retaining the original values and source history."""
    out = raw.copy()
    if out.empty:
        out = out.reindex(columns=list(dict.fromkeys(list(out.columns) + ["lot_transn_seq", "slot_id", "transn_occur_date", "data_kind", "test_point_no", "coordinate_x", "coordinate_y", "value_x", "value_y", "mrc_x_valn", "mrc_y_valn"])))
    require_columns(out, ["lot_transn_seq", "data_kind", "value_x", "value_y"], "OCO RAW SOURCE")
    if not np.isfinite(scale) or scale < 0:
        raise ValueError("OCO edge 배율은 유한한 0 이상의 수여야 합니다.")
    measured = out["data_kind"].eq("RAWDATA")
    out["measurement_source"], out["source_mstepseq"] = source_name, mstepseq
    out["source_lot_transn_seq"] = integer(out["lot_transn_seq"])
    out["measurement_scale"] = np.where(measured, float(scale), 1.0)
    for axis in ["x", "y"]:
        col, original = f"value_{axis}", f"value_{axis}_original"
        # DB results may be categorical or nullable integer; use float before replacement.
        current = pd.to_numeric(out[col], errors="coerce").astype(float)
        # Reusing an already-prepared frame never applies the factor a second time.
        values = pd.to_numeric(out[original], errors="coerce").astype(float) if original in out else current
        out[original] = values.where(measured)
        out[col] = current.where(~measured, values * scale)
    return out


sql_rawdata_oco = rawdata_sql(cfg, df_lotinfo_oco) if not df_lotinfo_oco.empty else None
sql_rawdata_adi = rawdata_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
edge_analysis_group = "E2" if (cfg.target_mstepseq_oco or "")[2:5] == "077" else "E1"
sql_rawdata_oco_edge = rawdata_sql(cfg, df_lotinfo_oco_edge, test_group=edge_analysis_group) if not df_lotinfo_oco_edge.empty else None
df_rawdata_oco_main_query = fetch_frame("rawdata_oco", sql_rawdata_oco, reuse=False) if sql_rawdata_oco else pd.DataFrame()
df_rawdata_oco_main_query = prepare_oco_raw_source(df_rawdata_oco_main_query, "oco_main", cfg.target_mstepseq_oco)
df_rawdata_oco_edge_query = fetch_frame("rawdata_oco_edge", sql_rawdata_oco_edge, reuse=False) if sql_rawdata_oco_edge else pd.DataFrame()
if not df_rawdata_oco_edge_query.empty:
    lower, upper = (81, 160) if edge_analysis_group == "E2" else (1, 80)
    df_rawdata_oco_edge_query = df_rawdata_oco_edge_query[pd.to_numeric(df_rawdata_oco_edge_query["test_point_no"], errors="coerce").between(lower, upper)].copy()
df_rawdata_oco_edge_query = prepare_oco_raw_source(df_rawdata_oco_edge_query, "oco_edge", cfg.target_mstepseq_oco_edge, cfg.oco_edge_scale)
oco_raw_parts = [frame for frame in [df_rawdata_oco_main_query, df_rawdata_oco_edge_query] if not frame.empty]
df_rawdata_oco_query = pd.concat(oco_raw_parts, ignore_index=True) if oco_raw_parts else df_rawdata_oco_main_query.copy()
df_rawdata_adi_query = fetch_frame("rawdata_adi", sql_rawdata_adi, reuse=False) if sql_rawdata_adi else pd.DataFrame()
save_stage("05_RAWDATA_OCO_MAIN_QUERY", df_rawdata_oco_main_query)
save_stage("05_RAWDATA_OCO_EDGE_SCALED", df_rawdata_oco_edge_query)
save_stage("05_RAWDATA_OCO_QUERY", df_rawdata_oco_query)
save_stage("05_RAWDATA_ADI_QUERY", df_rawdata_adi_query)


## 06. 교체할 코드
OCO는 lot_transn_seq·data_kind·TEST 그룹별로 최신 batch를 선택해 E2의 늦은 측정 때문에 E1이 사라지지 않게 합니다. ADI batch 선택은 기존대로 유지합니다.


In [ ]:
def latest_raw_batch(raw, lots, gap_seconds=60, separate_test_groups=False):
    if raw.empty or lots.empty:
        return raw.iloc[:0].copy()
    require_columns(raw, ["lot_transn_seq", "slot_id", "data_kind", "transn_occur_date"], "RAWDATA")
    df = raw.copy()
    df["transn_occur_date"] = pd.to_datetime(df["transn_occur_date"].astype(str), errors="coerce")
    df["slot_id"] = integer(df["slot_id"])
    df = df.dropna(subset=["slot_id", "transn_occur_date"]).copy()
    df["slot_id"] = df["slot_id"].astype(int)
    keys = lots[["lot_transn_seq", "slotid"]].drop_duplicates().copy()
    keys["lot_transn_seq"], keys["slotid"] = integer(keys["lot_transn_seq"]), integer(keys["slotid"])
    df["lot_transn_seq"] = integer(df["lot_transn_seq"])
    df = df.merge(keys.dropna(), left_on=["lot_transn_seq", "slot_id"], right_on=["lot_transn_seq", "slotid"], how="inner")
    groups = ["lot_transn_seq", "slot_id", "data_kind"]
    if separate_test_groups:
        require_columns(df, ["test_point_no"], "OCO TEST group batch")
        df["_measurement_group"] = ((pd.to_numeric(df["test_point_no"], errors="coerce") - 1) // 80).astype("Int64")
        groups.append("_measurement_group")
    df = df.sort_values(groups + ["transn_occur_date"]).reset_index(drop=True)
    previous = df.groupby(groups, observed=True, dropna=False)["transn_occur_date"].shift()
    starts = previous.isna() | (df["transn_occur_date"] - previous).dt.total_seconds().gt(gap_seconds)
    df["_batch"] = starts.astype(int).groupby([df[col] for col in groups], observed=True, dropna=False).cumsum()
    # transform으로 최대 batch를 붙이면 원본 merge와 같은 행 선택을 중간 테이블 없이 수행합니다.
    last = df.groupby(groups, observed=True, dropna=False)["_batch"].transform("max")
    return df[df["_batch"].eq(last)].drop(columns=["_batch", "_measurement_group"], errors="ignore").reset_index(drop=True)

df_rawdata_oco = latest_raw_batch(df_rawdata_oco_query, df_lotinfo_oco_all, cfg.rawdata_gap_seconds, separate_test_groups=True)
df_rawdata_adi = latest_raw_batch(df_rawdata_adi_query, df_lotinfo_adi, cfg.rawdata_gap_seconds)
save_stage("06_RAWDATA_OCO_LATEST", df_rawdata_oco)
save_stage("06_RAWDATA_ADI_LATEST", df_rawdata_adi)
source.release_cache()


## 08. 교체할 코드
기존 step E1은 K-MRC용으로 보존하고, 최종 분석 그룹(077: E2, 그 외: E1)도 보존합니다. 두 역할을 is_kmrc_reference/is_oco_analysis로 표시합니다.


In [ ]:
RENAME = {
    "pstepseq": "STEPSEQ", "lotid": "LOT_ID", "slotid": "Wafer", "peqpid": "P_EQPID", "ppid": "Photo_PPID",
    "chuckid": "ChuckID", "reticleid": "ReticleID", "steppitch_x": "STEP_PITCH_X", "steppitch_y": "STEP_PITCH_Y",
    "mapshift_x": "MAP_SHIFT_X", "mapshift_y": "MAP_SHIFT_Y", "dmargin_x": "Dmargin_X", "dmargin_y": "Dmargin_Y",
    "outlr_resdl_spec_ratio": "Outlier_Spec_Ratio", "chip_x_qty": "CHIP_X_NUM", "chip_y_qty": "CHIP_Y_NUM",
    "mmo_mrc_ref_eqp_id": "MMO_MRC_EQP", "mstepseq": "M_STEPSEQ", "timestamp": "M_TIME",
}

OCO_PROVENANCE_COLUMNS = ["measurement_source", "source_mstepseq", "source_lot_transn_seq", "measurement_scale", "X_reg_original", "Y_reg_original", "is_kmrc_reference", "is_oco_analysis"]

FINAL_COLUMNS = [
    "UNIQUE_ID", "UNIQUE_ID2", "UNIQUE_ID3", "UNIQUE_ID4", "STEPSEQ", "LOT_ID", "Wafer",
    "P_EQPID", "Photo_PPID", "P_TIME", "M_TIME", "ChuckID", "ReticleID", "Base_EQP1", "MMO_MRC_EQP", "GROUP", "TEST", "DieX", "DieY",
    "STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y", "coordinate_X", "coordinate_Y",
    "fcp_x", "fcp_y", "wf_x", "wf_y", "radius", "CHIP_X_NUM", "CHIP_Y_NUM", "Outlier_Spec_Ratio", "Dmargin_X", "Dmargin_Y",
    "X_reg", "Y_reg", "MRC_RX", "MRC_RY", "PSM_X", "PSM_Y", "Is_Edge_Shot", "Max_Corner_Distance",
    "M_STEPSEQ", "photo_transn_seq", "apc_hist_index_no", "apc_trocs_hist_index_no",
] + OCO_PROVENANCE_COLUMNS

def assemble_measurements(raw, lots, expo, f1, mode, cfg):
    if raw.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = raw[raw["data_kind"].eq("RAWDATA")].rename(columns={"test_point_no": "TEST", "coordinate_x": "DieX", "coordinate_y": "DieY", "value_x": "X_reg", "value_y": "Y_reg", "value_x_original": "X_reg_original", "value_y_original": "Y_reg_original"}).copy()
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = clean_columns(df).merge(lots, left_on=["lot_transn_seq", "slot_id"], right_on=["lot_transn_seq", "slotid"], how="left", validate="m:1", suffixes=("", "_lotinfo"))
    test = raw[raw["data_kind"].eq("TESTDATA")].rename(columns={"test_point_no": "TEST", "value_x": "coordinate_X", "value_y": "coordinate_Y", "mrc_x_valn": "MRC_RX", "mrc_y_valn": "MRC_RY"})
    if not test.empty:
        columns = ["lot_transn_seq", "slot_id", "TEST", "coordinate_X", "coordinate_Y", "MRC_RX", "MRC_RY"]
        df = df.merge(test[columns], on=columns[:3], how="left", validate="m:1", suffixes=("", "_test"))
    for col in ["coordinate_X", "coordinate_Y", "MRC_RX", "MRC_RY"]:
        if col not in df:
            df[col] = np.nan
    if mode == "ADI":
        pershot = raw[raw["data_kind"].eq("PERSHOT")].rename(columns={"test_point_no": "TEST", "coordinate_x": "DieX", "coordinate_y": "DieY", "value_x": "PSM_X", "value_y": "PSM_Y"})
        columns = ["lot_transn_seq", "slot_id", "TEST", "DieX", "DieY", "PSM_X", "PSM_Y"]
        if not pershot.empty:
            df = df.merge(pershot[columns], on=columns[:5], how="left", validate="m:1", suffixes=("", "_psm"))
        for col in ["PSM_X", "PSM_Y"]:
            if col not in df:
                df[col] = np.nan
    if not expo.empty:
        columns = ["lotid", "slot_id", "photo_transn_seq", "photo_date", "apc_hist_index_no", "apc_trocs_hist_index_no", "mmo_mrc_ref_eqp_id"]
        df = df.merge(expo[columns].rename(columns={"photo_date": "P_TIME", "photo_transn_seq": "photo_transn_seq_expo"}), on=["lotid", "slot_id"], how="left", validate="m:1")
    else:
        for col in ["P_TIME", "apc_hist_index_no", "apc_trocs_hist_index_no", "mmo_mrc_ref_eqp_id"]:
            df[col] = pd.NA
    # 원본의 photo_transn_seq_x/y 충돌 때문에 빠지던 F1 메타데이터만 명시적으로 연결합니다.
    if not f1.empty:
        f1 = f1.copy()
        f1["photo_transn_seq"], f1["slotid"] = integer(f1["photo_transn_seq"]), integer(f1["slotid"])
        f1["_f1_lot_key"] = f1["lotid"].astype(str).str.strip()
        df["_f1_lot_key"] = df["lotid"].astype(str).str.strip()
        keys = ["photo_transn_seq", "_f1_lot_key", "slotid"]
        f1 = f1.sort_values("impala_insert_time").drop_duplicates(keys, keep="last") if "impala_insert_time" in f1 else f1.drop_duplicates(keys, keep="last")
        df = df.merge(f1[keys + ["base_eqp_id1"]], on=keys, how="left", validate="m:1").rename(columns={"base_eqp_id1": "Base_EQP1"})
        df = df.drop(columns="_f1_lot_key")
    df = clean_columns(df.rename(columns=RENAME)).reset_index(drop=True)
    numeric(df, ["TEST", "X_reg", "Y_reg", "PSM_X", "PSM_Y", "MRC_RX", "MRC_RY"])
    df["GROUP"] = pd.cut(df["TEST"], bins=[0, 80, 160, 240], labels=["E1", "E2", "E3"])
    if mode == "OCO":
        group = "E2" if (cfg.target_mstepseq_oco or "")[2:5] == "077" else "E1"
        source = df["measurement_source"] if "measurement_source" in df else pd.Series("oco_main", index=df.index)
        keep_reference = df["GROUP"].eq("E1") & source.eq("oco_main")
        df["is_kmrc_reference"], df["is_oco_analysis"] = keep_reference, df["GROUP"].eq(group)
        df = df[keep_reference | df["is_oco_analysis"]].copy()
    return add_geometry_and_ids(df, cfg)

def add_geometry_and_ids(df, cfg):
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = df.reset_index(drop=True)
    columns = ["DieX", "DieY", "STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y", "coordinate_X", "coordinate_Y"]
    require_columns(df, columns, "좌표 계산")
    for col in columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
    df["fcp_x"] = df["DieX"] * df["STEP_PITCH_X"] + df["MAP_SHIFT_X"]
    df["fcp_y"] = df["DieY"] * df["STEP_PITCH_Y"] + df["MAP_SHIFT_Y"]
    df["wf_x"], df["wf_y"] = df["fcp_x"] + df["coordinate_X"], df["fcp_y"] + df["coordinate_Y"]
    df["radius"] = np.sqrt(df["wf_x"] ** 2 + df["wf_y"] ** 2)
    fields = ["STEPSEQ", "P_EQPID", "Photo_PPID", "MMO_MRC_EQP", "P_TIME", "M_TIME", "LOT_ID", "Wafer"]
    text = {col: df[col].astype("string").fillna("") for col in fields + ["GROUP", "TEST", "DieX", "DieY"]}
    base = text[fields[0]]
    for col in fields[1:]:
        base = base + "_" + text[col]
    df["UNIQUE_ID3"], df["UNIQUE_ID"] = base, base + "_" + text["GROUP"]
    df["UNIQUE_ID2"] = base + "_" + text["TEST"] + "_" + text["DieX"] + "_" + text["DieY"] + "_" + text["GROUP"]
    df["UNIQUE_ID4"] = base + "_" + text["DieX"] + "_" + text["DieY"] + "_" + text["GROUP"]
    if "measurement_source" in df:
        edge = df["measurement_source"].eq("oco_edge")
        suffix = "_OCO_EDGE_" + integer(df["source_lot_transn_seq"]).astype("string").fillna("unknown")
        for col in ["UNIQUE_ID", "UNIQUE_ID2", "UNIQUE_ID3", "UNIQUE_ID4"]:
            df.loc[edge, col] = df.loc[edge, col] + suffix.loc[edge]
    x, y = df["fcp_x"].to_numpy(float), df["fcp_y"].to_numpy(float)
    sx, sy = df["STEP_PITCH_X"].to_numpy(float) / 2, df["STEP_PITCH_Y"].to_numpy(float) / 2
    distances = [(x + dx * sx) ** 2 + (y + dy * sy) ** 2 for dx, dy in [(-1, -1), (-1, 1), (1, -1), (1, 1)]]
    df["Max_Corner_Distance"] = np.sqrt(np.maximum.reduce(distances))
    df["Is_Edge_Shot"] = df["Max_Corner_Distance"].gt(cfg.wafer_radius_um)
    return df[[col for col in FINAL_COLUMNS if col in df]].copy()

df_adi_measured = assemble_measurements(df_rawdata_adi, df_lotinfo_adi, df_expo_overlay, df_paramdata, "ADI", cfg)
df_oco_measured = assemble_measurements(df_rawdata_oco, df_lotinfo_oco_all, df_expo_overlay, df_paramdata, "OCO", cfg)
save_stage("08_ADI_MEASURED", df_adi_measured)
save_stage("08_OCO_MEASURED", df_oco_measured)

# OCO edge coverage after the existing group filter
def oco_edge_coverage(measured, history_status):
    keys = ["LOT_ID", "Wafer", "photo_transn_seq"]
    result = history_status.rename(columns={"lotid": "LOT_ID", "slotid": "Wafer"}).copy()
    for source_name, count_col in [("oco_main", "n_main_points"), ("oco_edge", "n_edge_points")]:
        analysis = measured["is_oco_analysis"] if "is_oco_analysis" in measured else pd.Series(True, index=measured.index)
        subset = measured[analysis & measured["measurement_source"].eq(source_name)]
        counts = subset.groupby(keys, observed=True, dropna=False).size().rename(count_col).reset_index()
        result = result.merge(counts, on=keys, how="left", validate="1:1")
        result[count_col] = result[count_col].fillna(0).astype(int)
    result["edge_point_status"] = np.where(result["n_edge_points"].gt(0), "included", np.where(result["edge_history_found"], "no_points_after_group_filter", result["edge_history_status"]))
    return result


df_oco_edge_coverage = oco_edge_coverage(df_oco_measured, df_oco_edge_history_status)
save_stage("08_OCO_EDGE_COVERAGE", df_oco_edge_coverage)


## 15. 교체할 코드
기존 MMO_MRC_EQP 필터 후 OCO 분석 raw와 E1 K-MRC raw를 별도 저장합니다. E1은 main step만 사용합니다. 단일 step에서만 OCO E1 반경 ≤144mm, ADI는 전체 측정점을 유지합니다.


In [ ]:
def summarize(frame, mode):
    if frame.empty:
        return pd.DataFrame(columns=["mode", "LOT_ID", "Wafer", "rows", "mrc_matched", "psm_fit_matched", "valid_raw", "invalid_raw"])
    work = frame[["LOT_ID", "Wafer", "decorrect_valid", "mrc_status", "psm_fit_status"]].copy()
    work["mrc_matched"], work["psm_fit_matched"] = work["mrc_status"].eq("matched"), work["psm_fit_status"].eq("matched")
    out = work.groupby(["LOT_ID", "Wafer"], dropna=False, observed=True).agg(rows=("decorrect_valid", "size"), mrc_matched=("mrc_matched", "sum"), psm_fit_matched=("psm_fit_matched", "sum"), valid_raw=("decorrect_valid", "sum")).reset_index()
    out.insert(0, "mode", mode)
    out["invalid_raw"] = out["rows"] - out["valid_raw"]
    return out

def select_oco_model_inputs(frame, cfg):
    """Keep E1 main references separate from full-area CPE input; never restrict ADI here."""
    group = "E2" if (cfg.target_mstepseq_oco or "")[2:5] == "077" else "E1"
    source = frame["measurement_source"] if "measurement_source" in frame else pd.Series("oco_main", index=frame.index)
    reference_mask = frame["GROUP"].eq("E1") & source.eq("oco_main")
    split_steps = bool(cfg.include_oco_edge and cfg.target_mstepseq_oco_edge != cfg.target_mstepseq_oco)
    references = frame.loc[reference_mask].copy()
    if not split_steps and not references.empty:
        require_columns(references, ["radius"], "OCO E1 K-MRC 반경")
        radius = pd.to_numeric(references["radius"], errors="coerce")
        references = references[np.isfinite(radius) & radius.le(cfg.kmrc_e1_radius_um)].copy()
    analysis = frame[frame["GROUP"].eq(group) & (source.eq("oco_main") | split_steps)].copy()
    keys = ["LOT_ID", "Wafer"]
    counts = frame[keys].drop_duplicates().copy()
    for name, part in [("n_e1_main_before_radius", frame.loc[reference_mask]), ("n_e1_kmrc", references), ("n_analysis", analysis)]:
        grouped = part.groupby(keys, observed=True, dropna=False).size().rename(name).reset_index()
        counts = counts.merge(grouped, on=keys, how="left", validate="1:1")
        counts[name] = counts[name].fillna(0).astype(int)
    counts["analysis_group"], counts["split_steps"] = group, split_steps
    counts["e1_radius_limit_um"] = np.nan if split_steps else cfg.kmrc_e1_radius_um
    return analysis, references, counts


df_final_adi = df_adi_raw[df_adi_raw["MMO_MRC_EQP"].notna() & df_adi_raw["MMO_MRC_EQP"].astype(str).str.strip().ne("")].copy()
df_final_oco_all = df_oco_raw[df_oco_raw["MMO_MRC_EQP"].notna() & df_oco_raw["MMO_MRC_EQP"].astype(str).str.strip().ne("")].copy()
df_final_oco, df_oco_e1_raw, df_oco_model_input_counts = select_oco_model_inputs(df_final_oco_all, cfg)
df_wafer_summary = pd.concat([summarize(df_final_adi, "ADI"), summarize(df_final_oco, "OCO")], ignore_index=True)
save_stage("15_ADI_FINAL", df_final_adi, columns=raw_preview)
save_stage("15_OCO_FINAL", df_final_oco, columns=raw_preview)
save_stage("15_OCO_ALL_GROUPS_RAW", df_final_oco_all, columns=raw_preview)
save_stage("15_OCO_E1_KMRC_INPUT", df_oco_e1_raw, columns=raw_preview)
save_stage("15_OCO_MODEL_INPUT_COUNTS", df_oco_model_input_counts)
save_stage("15_WAFER_SUMMARY", df_wafer_summary)
print("MRC/PSM decorrect 완료: OCO 분석 입력과 E1 K-MRC 입력을 별도로 저장했습니다.")


## 20. 교체할 코드
기존 축당 19계수 WK/RK 모델을 사용합니다. K-MRC = 회귀(OCO main E1 raw) − 회귀(ADI raw). OCO 분석 위치에서 이 차이 계수를 평가하는 함수를 추가했습니다.


In [ ]:
WK_RK_MIN_POINTS = 19

def osr_wk20p_rk20p(x, y, rx, ry):
    """
    회사 코드의 osr_wk20p_rk20p 구조를 그대로 사용.
    x, y   : wafer/field level 좌표 -> fcp_x, fcp_y
    rx, ry : shot 내부 좌표         -> coordinate_X, coordinate_Y
    반환:
        X_dx: X 방향 회귀용 design matrix (N x 19)
        X_dy: Y 방향 회귀용 design matrix (N x 19)
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(x)),
        x / 1e6, -y / 1e6,
        (x ** 2) / 1e12, (x * y) / 1e12, (y ** 2) / 1e12,
        (x ** 3) / 1e15, (x ** 2 * y) / 1e15, (x * y ** 2) / 1e15, (y ** 3) / 1e15,
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
    ]).T
    X_dy = np.vstack([
        np.ones(len(y)),
        y / 1e6, x / 1e6,
        (y ** 2) / 1e12, (y * x) / 1e12, (x ** 2) / 1e12,
        (y ** 3) / 1e15, (y ** 2 * x) / 1e15, (y * x ** 2) / 1e15, (x ** 3) / 1e15,
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12, (rx ** 3) / 1e12,
    ]).T
    return X_dx, X_dy
COEFF_KEYS_DX = [
    "WK1", "WK3", "WK5", "WK7", "WK9", "WK11", "WK13", "WK15", "WK17", "WK19",
    "RK3", "RK5", "RK7", "RK9", "RK11", "RK13", "RK15", "RK17", "RK19",
]
COEFF_KEYS_DY = [
    "WK2", "WK4", "WK6", "WK8", "WK10", "WK12", "WK14", "WK16", "WK18", "WK20",
    "RK4", "RK6", "RK8", "RK10", "RK12", "RK14", "RK16", "RK18", "RK20",
]
WK_COLS_DX = COEFF_KEYS_DX[:10]
RK_COLS_DX = COEFF_KEYS_DX[10:]
WK_COLS_DY = COEFF_KEYS_DY[:10]
RK_COLS_DY = COEFF_KEYS_DY[10:]

def normalize_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def fit_lstsq(X, y):
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ coef
    return coef, pred

def run_company_joint_regression(df, label="ADI", min_points=19):
    """
    wafer 단위로 raw_x/raw_y를 회사 룰 osr_wk20p_rk20p로 동시 회귀.
    입력:
      - raw_x, raw_y
      - fcp_x, fcp_y
      - coordinate_X, coordinate_Y
      - LOT_ID, Wafer
    출력:
      - wk_fit_x/y
      - rk_fit_x/y
      - total_fit_x/y
      - raw_minus_total_fit_x/y
      - wafer별 coefficient dataframe
    """
    df = df.copy()
    required_cols = [
        "LOT_ID", "Wafer",
        "raw_x", "raw_y",
        "fcp_x", "fcp_y",
        "coordinate_X", "coordinate_Y",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 필수 컬럼 누락: {missing}")
    normalize_numeric(df, [
        "raw_x", "raw_y",
        "fcp_x", "fcp_y",
        "coordinate_X", "coordinate_Y",
    ])
    result_cols = [
        "wk_fit_x", "wk_fit_y",
        "rk_fit_x", "rk_fit_y",
        "total_fit_x", "total_fit_y",
        "raw_minus_total_fit_x", "raw_minus_total_fit_y",
    ]
    for c in result_cols:
        df[c] = np.nan
    coef_rows = []
    summary_rows = []
    for (lot_id, wafer), grp in df.groupby(["LOT_ID", "Wafer"], dropna=False, observed=True):
        idx = grp.index
        valid_mask = (
            grp["raw_x"].notna() &
            grp["raw_y"].notna() &
            grp["fcp_x"].notna() &
            grp["fcp_y"].notna() &
            grp["coordinate_X"].notna() &
            grp["coordinate_Y"].notna()
        )
        work = grp.loc[valid_mask].copy()
        n_valid = len(work)
        if n_valid < min_points:
            summary_rows.append({
                "LOT_ID": lot_id,
                "Wafer": wafer,
                "n_total": len(grp),
                "n_valid": n_valid,
                "status": "skip_too_few_points",
            })
            continue
        x = work["fcp_x"].to_numpy(dtype=float)
        y = work["fcp_y"].to_numpy(dtype=float)
        rx = work["coordinate_X"].to_numpy(dtype=float)
        ry = work["coordinate_Y"].to_numpy(dtype=float)
        raw_x = work["raw_x"].to_numpy(dtype=float)
        raw_y = work["raw_y"].to_numpy(dtype=float)
        X_dx, X_dy = osr_wk20p_rk20p(x, y, rx, ry)
        coef_x, pred_x = fit_lstsq(X_dx, raw_x)
        coef_y, pred_y = fit_lstsq(X_dy, raw_y)
        wk_fit_x = X_dx[:, :10] @ coef_x[:10]
        rk_fit_x = X_dx[:, 10:] @ coef_x[10:]
        wk_fit_y = X_dy[:, :10] @ coef_y[:10]
        rk_fit_y = X_dy[:, 10:] @ coef_y[10:]
        total_fit_x = wk_fit_x + rk_fit_x
        total_fit_y = wk_fit_y + rk_fit_y
        residual_x = raw_x - total_fit_x
        residual_y = raw_y - total_fit_y
        valid_idx = work.index
        df.loc[valid_idx, "wk_fit_x"] = wk_fit_x
        df.loc[valid_idx, "rk_fit_x"] = rk_fit_x
        df.loc[valid_idx, "total_fit_x"] = total_fit_x
        df.loc[valid_idx, "raw_minus_total_fit_x"] = residual_x
        df.loc[valid_idx, "wk_fit_y"] = wk_fit_y
        df.loc[valid_idx, "rk_fit_y"] = rk_fit_y
        df.loc[valid_idx, "total_fit_y"] = total_fit_y
        df.loc[valid_idx, "raw_minus_total_fit_y"] = residual_y
        coef_row = {
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "n_total": len(grp),
            "n_valid": n_valid,
        }
        for name, val in zip(COEFF_KEYS_DX, coef_x):
            coef_row[f"{name}_X"] = val
        for name, val in zip(COEFF_KEYS_DY, coef_y):
            coef_row[f"{name}_Y"] = val
        coef_rows.append(coef_row)
        summary_rows.append({
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "n_total": len(grp),
            "n_valid": n_valid,
            "rms_raw_x": float(np.sqrt(np.nanmean(raw_x ** 2))),
            "rms_raw_y": float(np.sqrt(np.nanmean(raw_y ** 2))),
            "rms_residual_x": float(np.sqrt(np.nanmean(residual_x ** 2))),
            "rms_residual_y": float(np.sqrt(np.nanmean(residual_y ** 2))),
            "status": "ok",
        })
    coef_df = pd.DataFrame(coef_rows)
    summary_df = pd.DataFrame(summary_rows)
    print(f"[{label}] regression 완료")
    print(f"  - wafer 수: {df[['LOT_ID','Wafer']].drop_duplicates().shape[0]}")
    print(f"  - coefficient rows: {len(coef_df)}")
    print(f"  - summary rows: {len(summary_df)}")
    return df, coef_df, summary_df

# K-MRC difference model
def apply_kmrc_difference(analysis, adi_coeff, e1_coeff, adi_summary, e1_summary):
    """K-MRC = beta(OCO main E1 raw) - beta(ADI raw); evaluate at OCO analysis points."""
    keys = ["LOT_ID", "Wafer"]
    xcols, ycols = [f"{name}_X" for name in COEFF_KEYS_DX], [f"{name}_Y" for name in COEFF_KEYS_DY]
    values = ["wk_fit_x", "wk_fit_y", "rk_fit_x", "rk_fit_y", "total_fit_x", "total_fit_y", "raw_minus_total_fit_x", "raw_minus_total_fit_y"]
    out = analysis.copy().reset_index(drop=True)
    numeric(out, ["raw_x", "raw_y", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"])
    for col in values:
        out[col] = np.nan
    out["global_model_source"] = "KMRC_OCO_E1_RAW_MINUS_ADI_RAW"

    def lookup(frame):
        return {tuple(row[k] for k in keys): row for row in frame.to_dict("records")}

    adi, e1, adi_info, e1_info = map(lookup, [adi_coeff, e1_coeff, adi_summary, e1_summary])
    coefficients, diagnostics, accepted, rejected = [], [], [], []
    for key, grp in out.groupby(keys, observed=True, dropna=False, sort=False):
        a, e = adi.get(key), e1.get(key)
        ai, ei = adi_info.get(key, {}), e1_info.get(key, {})
        row = dict(zip(keys, key))
        row.update(n_total=len(grp), n_adi_valid=ai.get("n_valid", 0), n_oco_e1_valid=ei.get("n_valid", 0), n_parameters_per_axis=19)
        row["adi_fit_status"], row["oco_e1_fit_status"] = ai.get("status", "missing_adi"), ei.get("status", "missing_oco_e1")
        reason = "missing_or_insufficient_adi" if a is None else "missing_or_insufficient_oco_e1" if e is None else None
        if reason:
            row.update(status="excluded", exclusion_reason=reason, n_valid=0)
            diagnostics.append(row)
            rejected.append(grp.assign(kmrc_exclusion_reason=reason))
            continue
        bx = np.array([e[col] - a[col] for col in xcols], dtype=float)
        by = np.array([e[col] - a[col] for col in ycols], dtype=float)
        if not np.isfinite(bx).all() or not np.isfinite(by).all():
            row.update(status="excluded", exclusion_reason="nonfinite_kmrc", n_valid=0)
            diagnostics.append(row)
            rejected.append(grp.assign(kmrc_exclusion_reason="nonfinite_kmrc"))
            continue
        numeric_cols = ["raw_x", "raw_y", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"]
        valid = np.isfinite(grp[numeric_cols].to_numpy(float)).all(axis=1)
        work = grp.loc[valid]
        block = grp.copy()
        if len(work):
            mx, my = osr_wk20p_rk20p(*(work[col].to_numpy(float) for col in numeric_cols[2:]))
            for axis, matrix, beta in [("x", mx, bx), ("y", my, by)]:
                wk, rk = matrix[:, :10] @ beta[:10], matrix[:, 10:] @ beta[10:]
                block.loc[work.index, f"wk_fit_{axis}"] = wk
                block.loc[work.index, f"rk_fit_{axis}"] = rk
                block.loc[work.index, f"total_fit_{axis}"] = wk + rk
                block.loc[work.index, f"raw_minus_total_fit_{axis}"] = work[f"raw_{axis}"].to_numpy(float) - wk - rk
        row.update(status="ok", exclusion_reason="", n_valid=len(work))
        diagnostics.append(row)
        coeff_row = dict(row)
        coeff_row.update(zip(xcols, bx))
        coeff_row.update(zip(ycols, by))
        coefficients.append(coeff_row)
        accepted.append(block)
    base = keys + ["n_total", "n_valid", "n_adi_valid", "n_oco_e1_valid", "n_parameters_per_axis", "adi_fit_status", "oco_e1_fit_status", "status", "exclusion_reason"]
    result = pd.concat(accepted, ignore_index=True) if accepted else out.iloc[:0].copy()
    excluded = pd.concat(rejected, ignore_index=True) if rejected else out.iloc[:0].assign(kmrc_exclusion_reason=pd.Series(dtype="object"))
    coeff = pd.DataFrame(coefficients, columns=base + xcols + ycols)
    diag = pd.DataFrame(diagnostics, columns=base)
    return result, coeff, diag, excluded

def attach_measured_shot_indices(summary, points, require_all_shots=True):
    """Attach the measured shot grid index; (0, 0) is the map-shift origin, not the sampling mean."""
    keys = ["LOT_ID", "Wafer", "fcp_x", "fcp_y"]
    if summary.empty:
        if require_all_shots and not points.empty:
            raise ValueError("실측 shot이 있는데 shot 통계가 비어 있습니다.")
        result = summary.drop(columns=["DieX", "DieY", "shot_index_status"], errors="ignore").copy()
        for key in keys:
            result[key] = points[key].iloc[:0].reset_index(drop=True)
        result["DieX"], result["DieY"] = pd.Series(dtype="Int64"), pd.Series(dtype="Int64")
        result["shot_index_status"] = pd.Series(dtype="object")
        first = ["LOT_ID", "Wafer", "DieX", "DieY", "fcp_x", "fcp_y", "shot_index_status"]
        return result[first + [col for col in summary.columns if col not in first]]
    work = points[keys].copy()
    for axis, die in [("x", "DieX"), ("y", "DieY")]:
        values = pd.to_numeric(points[die], errors="coerce").astype(float) if die in points else pd.Series(np.nan, index=points.index)
        pitch_col, shift_col = f"STEP_PITCH_{axis.upper()}", f"MAP_SHIFT_{axis.upper()}"
        if pitch_col in points and shift_col in points:
            pitch = pd.to_numeric(points[pitch_col], errors="coerce").astype(float)
            shift = pd.to_numeric(points[shift_col], errors="coerce").astype(float)
            center = pd.to_numeric(points[f"fcp_{axis}"], errors="coerce").astype(float)
            derived = ((center - shift) / pitch.where(pitch.gt(0))).replace([np.inf, -np.inf], np.nan)
            usable = derived.notna() & np.isclose(derived, derived.round(), rtol=0, atol=1e-5)
            conflict = values.notna() & usable & ~np.isclose(values, derived.round(), rtol=0, atol=1e-5)
            if conflict.any():
                raise ValueError(f"{die}: 실측 index와 (fcp - MAP_SHIFT) / STEP_PITCH가 다릅니다. 좌표 메타데이터를 확인하세요.")
            values = values.fillna(derived.round().where(usable))
        invalid = values.notna() & (~np.isfinite(values) | ~np.isclose(values, values.round(), rtol=0, atol=1e-5))
        if invalid.any():
            raise ValueError(f"{die}: 정수가 아닌 shot index가 있습니다. 임의로 재번호를 부여하지 않습니다.")
        work[die] = values.round().astype("Int64")
    groups = work.groupby(keys, observed=True, dropna=False, sort=False)
    conflicts = groups[["DieX", "DieY"]].nunique(dropna=True).gt(1).any(axis=1)
    if conflicts.any():
        raise ValueError("같은 LOT/wafer/shot 중심에 여러 DieX/DieY가 있습니다. 노광 이력 또는 좌표를 확인하세요.")
    mapping = groups[["DieX", "DieY"]].first().reset_index()
    result = summary.drop(columns=["DieX", "DieY", "shot_index_status"], errors="ignore").merge(mapping, on=keys, how="left", validate="1:1", indicator="_shot_match")
    if result["_shot_match"].ne("both").any() or (require_all_shots and len(result) != len(mapping)):
        raise ValueError("Shot 통계의 key가 실측 point의 shot 목록과 다릅니다. 18번 summarize_values를 다시 실행하세요.")
    result = result.drop(columns="_shot_match")
    result["shot_index_status"] = np.where(result[["DieX", "DieY"]].notna().all(axis=1), "ok", "missing_geometry")
    if result["shot_index_status"].ne("ok").any():
        print("일부 shot의 DieX/DieY를 복원할 수 없습니다. 08/19 단계의 index·pitch·map shift를 확인하세요.")
    first = ["LOT_ID", "Wafer", "DieX", "DieY", "fcp_x", "fcp_y", "shot_index_status"]
    return result[first + [col for col in summary.columns if col not in first]]


## 21. 교체할 코드
ADI 자체 회귀는 유지합니다. OCO는 E1−ADI의 K-MRC를 분석 raw에서 제거하고 raw_minus_total_fit을 만듭니다. 기준 회귀가 불가능한 wafer는 진단·제외 자료에 기록하고 OCO 후속 CPE에서 제외합니다.


In [ ]:
def split_wk_rk_coefficients(coeff):
    base = ["LOT_ID", "Wafer", "n_total", "n_valid"]
    wk = base + [f"{key}_{axis}" for axis, names in [("X", WK_COLS_DX), ("Y", WK_COLS_DY)] for key in names]
    rk = base + [f"{key}_{axis}" for axis, names in [("X", RK_COLS_DX), ("Y", RK_COLS_DY)] for key in names]
    return coeff.reindex(columns=wk), coeff.reindex(columns=rk)

def wk_rk_rank_diagnostics(frame, summary):
    rank_columns = WAFER_KEYS + ["rank_x", "rank_y", "n_parameters_per_axis", "rank_deficient"]
    if frame.empty:
        return summary.reindex(columns=list(dict.fromkeys(list(summary.columns) + rank_columns)))
    rows = []
    cols = ["raw_x", "raw_y", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"]
    for key, grp in frame.groupby(WAFER_KEYS, dropna=False, observed=True):
        valid = numeric(grp[cols].copy(), cols).dropna()
        if not valid.empty:
            mx, my = osr_wk20p_rk20p(*(valid[col].to_numpy(dtype=float) for col in cols[2:]))
            rank_x, rank_y = np.linalg.matrix_rank(mx), np.linalg.matrix_rank(my)
        else:
            rank_x, rank_y = 0, 0
        rows.append({"LOT_ID": key[0], "Wafer": key[1], "rank_x": rank_x, "rank_y": rank_y, "n_parameters_per_axis": 19, "rank_deficient": rank_x < 19 or rank_y < 19})
    rank = pd.DataFrame(rows, columns=rank_columns)
    return summary.reindex(columns=list(dict.fromkeys(WAFER_KEYS + list(summary.columns)))).merge(rank, on=WAFER_KEYS, how="left", validate="1:1")

df_adi_reg, coef_adi, summary_adi = run_company_joint_regression(df_adi_trocs, label="ADI", min_points=WK_RK_MIN_POINTS)
df_oco_e1_reg, coef_oco_e1, summary_oco_e1 = run_company_joint_regression(df_oco_e1_raw, label="OCO_E1_KMRC_REFERENCE", min_points=WK_RK_MIN_POINTS)
df_oco_reg, coef_oco, summary_oco, df_oco_kmrc_excluded = apply_kmrc_difference(df_oco_trocs, coef_adi, coef_oco_e1, summary_adi, summary_oco_e1)
df_wk_coeff_adi, df_rk_coeff_adi = split_wk_rk_coefficients(coef_adi)
df_wk_coeff_oco, df_rk_coeff_oco = split_wk_rk_coefficients(coef_oco)
df_wk_rk_diag_adi = wk_rk_rank_diagnostics(df_adi_reg, summary_adi)
df_oco_e1_diag = wk_rk_rank_diagnostics(df_oco_e1_reg, summary_oco_e1)
df_wk_rk_diag_oco = summary_oco.copy()
for prefix, rank_source in [("adi", df_wk_rk_diag_adi), ("oco_e1", df_oco_e1_diag)]:
    cols = WAFER_KEYS + ["rank_x", "rank_y", "rank_deficient"]
    rank_source = rank_source.reindex(columns=cols).rename(columns={col: f"{prefix}_{col}" for col in cols if col not in WAFER_KEYS})
    df_wk_rk_diag_oco = df_wk_rk_diag_oco.merge(rank_source, on=WAFER_KEYS, how="left", validate="1:1")
wk_rk_values = ["raw_x", "raw_y", "wk_fit_x", "wk_fit_y", "rk_fit_x", "rk_fit_y", "raw_minus_total_fit_x", "raw_minus_total_fit_y"]
df_wk_rk_wafer_adi, df_wk_rk_wafer_oco = summarize_values(df_adi_reg, WAFER_KEYS, wk_rk_values), summarize_values(df_oco_reg, WAFER_KEYS, wk_rk_values)
df_wk_rk_shot_adi = attach_measured_shot_indices(summarize_values(df_adi_reg, SHOT_KEYS, wk_rk_values), df_adi_reg)
df_wk_rk_shot_oco = attach_measured_shot_indices(summarize_values(df_oco_reg, SHOT_KEYS, wk_rk_values), df_oco_reg)
df_wk_removed_adi = df_adi_reg[[col for col in POINT_KEYS + ["wk_fit_x", "wk_fit_y"] if col in df_adi_reg]].copy()
df_wk_removed_oco = df_oco_reg[[col for col in POINT_KEYS + ["wk_fit_x", "wk_fit_y"] if col in df_oco_reg]].copy()
df_rk_removed_adi = df_adi_reg[[col for col in POINT_KEYS + ["rk_fit_x", "rk_fit_y"] if col in df_adi_reg]].copy()
df_rk_removed_oco = df_oco_reg[[col for col in POINT_KEYS + ["rk_fit_x", "rk_fit_y"] if col in df_oco_reg]].copy()
wk_rk_results = [
    ("ADI", df_adi_reg, df_wk_coeff_adi, df_rk_coeff_adi, df_wk_removed_adi, df_rk_removed_adi, df_wk_rk_diag_adi, df_wk_rk_wafer_adi, df_wk_rk_shot_adi),
    ("OCO", df_oco_reg, df_wk_coeff_oco, df_rk_coeff_oco, df_wk_removed_oco, df_rk_removed_oco, df_wk_rk_diag_oco, df_wk_rk_wafer_oco, df_wk_rk_shot_oco),
]
for mode, reg, wk, rk, wk_points, rk_points, diag, wafer, shot in wk_rk_results:
    save_stage(f"21_{mode}_WK_RK_REMOVED", reg, columns=POINT_KEYS + wk_rk_values)
    save_stage(f"21_{mode}_WK_WAFER_COEFFICIENTS", wk)
    save_stage(f"21_{mode}_RK_WAFER_COEFFICIENTS", rk)
    save_stage(f"21_{mode}_WK_REMOVED_POINTS", wk_points)
    save_stage(f"21_{mode}_RK_REMOVED_POINTS", rk_points)
    save_stage(f"21_{mode}_WK_RK_WAFER_DIAGNOSTICS", diag)
    save_stage(f"21_{mode}_WK_RK_WAFER_DATA", wafer)
    save_stage(f"21_{mode}_WK_RK_SHOT_DATA", shot)
save_stage("21_OCO_E1_WAFER_COEFFICIENTS", coef_oco_e1)
save_stage("21_OCO_E1_WAFER_DIAGNOSTICS", df_oco_e1_diag)
save_stage("21_KMRC_WAFER_COEFFICIENTS", coef_oco)
save_stage("21_KMRC_EXCLUDED_OCO_POINTS", df_oco_kmrc_excluded)
print(f"K-MRC 적용: OCO {len(df_oco_reg):,} points, 제외 {len(df_oco_kmrc_excluded):,} points")


## 28. 교체할 코드
최종 분해식과 K-MRC·step·배율·반경 설정을 저장합니다. OCO의 WK/RK 제거값은 E2 자체 회귀값이 아니라 K-MRC fitting값입니다.


In [ ]:
def check_decomposition(frame, mode):
    checks = []
    for axis in ["x", "y"]:
        cols = [f"raw_{axis}", f"wk_fit_{axis}", f"rk_fit_{axis}", f"CPE_fit_{axis}", f"real_residual_{axis}"]
        valid = frame[cols].dropna()
        expected = valid[cols[1:]].sum(axis=1)
        error = (valid[cols[0]] - expected).abs()
        passed = np.isclose(valid[cols[0]], expected, rtol=1e-10, atol=1e-12)
        checks.append({"mode": mode, "axis": axis, "n_total": len(frame), "n_checked": len(valid), "n_unchecked": len(frame) - len(valid), "max_abs_error": float(error.max()) if len(error) else np.nan, "passed": bool(passed.all()) if len(valid) else False, "check_status": "checked" if len(valid) else "no_valid_points"})
        if len(valid) and not passed.all():
            raise ValueError(f"{mode}/{axis}: raw 분해식 검증 실패. 입력과 제거값을 확인하세요.")
    return pd.DataFrame(checks)

df_decomposition_check = pd.concat([check_decomposition(df_adi_cpe, "ADI"), check_decomposition(df_oco_cpe, "OCO")], ignore_index=True)
final_values = ["raw_x", "raw_y", "trocs_fit_x", "trocs_fit_y", "wk_fit_x", "wk_fit_y", "rk_fit_x", "rk_fit_y", "CPE_fit_x", "CPE_fit_y", "real_residual_x", "real_residual_y"]
df_final_wafer_adi, df_final_wafer_oco = summarize_values(df_adi_cpe, WAFER_KEYS, final_values), summarize_values(df_oco_cpe, WAFER_KEYS, final_values)
df_final_shot_adi, df_final_shot_oco = attach_measured_shot_indices(summarize_values(df_adi_cpe, SHOT_KEYS, final_values), df_adi_cpe), attach_measured_shot_indices(summarize_values(df_oco_cpe, SHOT_KEYS, final_values), df_oco_cpe)
save_stage("28_DECOMPOSITION_CHECK", df_decomposition_check)
save_stage("28_ADI_FINAL_WAFER_DATA", df_final_wafer_adi)
save_stage("28_OCO_FINAL_WAFER_DATA", df_final_wafer_oco)
save_stage("28_ADI_FINAL_SHOT_DATA", df_final_shot_adi)
save_stage("28_OCO_FINAL_SHOT_DATA", df_final_shot_oco)
extension_settings = dict(
    kmrc_definition="OCO_E1_RAW_MINUS_ADI_RAW", kmrc_parameters_per_axis=19,
    kmrc_e1_radius_um=cfg.kmrc_e1_radius_um, kmrc_adi_radius_limit_um=None, oco_split_steps=bool(cfg.include_oco_edge and cfg.target_mstepseq_oco_edge != cfg.target_mstepseq_oco),
    oco_main_step=cfg.target_mstepseq_oco, oco_edge_step=cfg.target_mstepseq_oco_edge, oco_edge_scale=cfg.oco_edge_scale,
    trocs_wafer_radius_um=TROCS_WAFER_RADIUS, trocs_subtracted_from_raw=False,
    trocs_query_file=str(TROCS_QUERY_FILE) if TROCS_QUERY_FILE is not None else None,
    wk_rk_min_points=WK_RK_MIN_POINTS, wk_rk_group_keys=WAFER_KEYS, cpe_group_keys=SHOT_KEYS, cpe_options=cpe_options,
    use_virtual_chip_points=USE_VIRTUAL_CHIP_POINTS, virtual_wafer_radius_um=VIRTUAL_WAFER_RADIUS,
    span_tolerance=SPAN_TOLERANCE, minimum_ridge=MINIMUM_RIDGE, add_ridge=ADD_RIDGE,
    max_gopt_return=MAX_GOPT_RETURN, max_span_iter=MAX_SPAN_ITER, cd_tol=CD_TOL,
    ridge_valves=RIDGE_VALVE_HYPER, active_span_limits=get_active_span_limits(SPAN_MODE),
)
config_path = OUTPUT_DIR / "extension_settings.json"
config_path.write_text(json.dumps(extension_settings, ensure_ascii=False, indent=2), encoding="utf-8")
display(FileLink(os.path.relpath(config_path, Path.cwd()), result_html_prefix="추가 단계 실행 설정: "))
source.release_cache()
print("TROCS 연결 → WK/RK 제거 → shot CPE38 제거까지 완료했습니다.")
